In [1]:
#NOTE THAT I DIDNT DO THE WIEGHTS YET HERE, DIDNT TALK TO MINA YET !!!!!!!!!!!!!!!!!!!!!!!!!!!!
import numpy as np  
import pandas as pd
#note total number of events is 8400000 
#total sum of weigths is 5207013978.837081   
 
import uproot as ur    
 
#for reading in the dataset, only having the phi track beacsue of storage space. 
 
#file = ur.open(r"C:\Users\keega\Downloads\com_8160_minpt_0_merged_tracks (1).root")
#file = ur.open(r"C:\Users\keega\Downloads\com_8160_minpt_5_merged_tracks.root") #new stuff with higher energiess 
file = ur.open(r"C:\Users\keega\Downloads\com_8160_minpt_5_merged.root") #bigger new stufff

# List all keys (e.g., trees, histograms)
#print(file.keys())   

# Access a TTree 
startt = 6800001
endd = 8400000
tree = file["jet_tree"]
vtrackphi  = tree['vtrackphi'].array(entry_start = startt, entry_stop = endd)  # Replace with actual tree name
# weights = tree['weight'].array(entry_start = startt, entry_stop = endd)
vtrackpt  = tree['vtrackpt'].array(entry_start = startt, entry_stop = endd)

In [10]:
800001+800000

1600001

In [ ]:
import numpy as np  
import pandas as pd
 
import uproot as ur    
#file = ur.open(r"C:\Users\keega\Downloads\com_8160_minpt_5_merged_tracks.root")
file = ur.open(r"C:\Users\keega\Downloads\com_8160_minpt_5_merged.root") #bigger new stufff

# List all keys (e.g., trees, histograms)
#print(file.keys())   

# Access a TTree 
#startt = 1001
#endd = 2000
tree = file["jet_tree"]
#vtrackphi  = tree['vtrackphi'].array(entry_start = startt, entry_stop = endd)  # Replace with actual tree name
weights = tree['weight'].array()
#vtrackpt  = tree['vtrackpt'].array(entry_start = startt, entry_stop = endd)
np.mean(weights)

In [6]:
np.sum(weights) #sum, 5207013978.837081, mean for this one, 619.8826165282239, mean for smaller one, #620.404832147202

np.float64(5207013978.837081)

In [2]:
# simple comulatans
#equation 4 
def Qmoment(a, n): 
    return np.sum(np.exp(1j*n*a))

#equation 16
def cumulant_2(a, n):  
    M= len(a)  
    if M<=1:
        return 0
    return (np.abs(Qmoment(a, n))**2-M)/( (M-1)*M ) 
    
#same as above, but with moments, equation 18 in the same paper. 


def cumulant_4(a, n):
    M = len(a)
    if M<=3:
        return 0
    Q2 = Qmoment(a, n)
    Q4 = Qmoment(a, 2*n)
    top = np.abs(Q2)**4+np.abs(Q4)**2-2*np.real(Q4* np.conjugate(Q2)*np.conjugate(Q2))
    bottom = 2*(M-2) *np.abs(Q2)**2-M*(M-3)
    normm = M*(M-1)*(M-2)*(M-3) 
    return (top-2*bottom)/normm

In [3]:
#POI defined
def Qmoment(a, n): #standard Q momement from last time
    return np.sum(np.exp(1j*n*a))

def Pmoment(a, n): #equation 27, note than pna nd qn are the same for this case
    return np.sum(np.exp(1j*n*a))

#i think qn (eq 27) is zero heree 
def POI_order2(M, mp, n):# n is the order, M is the ref particles, mp is POI particles
    return Pmoment(mp, n)*np.conjugate(Qmoment(M, n))/(len(mp)*(len(M)+len(mp))) 

def POI_order4(ROI, POI, n): # same as abovee 
    #M_total = len(ROI)+len(mp)
    M = len(ROI)
    mp = len(POI)
    pn = Pmoment(POI, n)
    qn=0
    Qn = Qmoment(ROI, n)
    Q2n = Qmoment(ROI, 2*n)
    if mp==0 or M<3:
        return 0
    top = (pn*Qn*np.conjugate(Qn)*np.conjugate(Qn) -pn*Qn*np.conjugate(Q2n)-2*M*pn*np.conjugate(Qn)
    +2*pn*np.conjugate(Qn)) #equation 32, qn, = 0
    norm = mp*M *(M-1)* (M-2 )
    return top/norm
def POI_order2(ROI, POI, n):
    M = len(ROI)
    mp = len(POI) 
    if M==0 or mp==0:
        return 0
    pn = Pmoment(POI, n)
    Qn = Qmoment(ROI, n)
    return pn*np.conjugate(Qn)/(M*mp) #equation 28, mq is zero 

In [4]:
#sample code to run itt.. where poi is greater than 3 gev 

length = len(vtrackpt) #should be 400000, but dont want to be off by like one

cumulant_moment_POI2 = np.zeros(length, dtype = complex)
cumulant_moment_POI4 = np.zeros(length, dtype = complex)
cumulant_moment_2 = np.zeros(length)
cumulant_moment_4 = np.zeros(length)
#sum_weight = np.sum(weights )  
import time
start_time = time.time() 
for i in range(length ): 
    phi = np.array(vtrackphi[i]) 
    pt = np.array(vtrackpt[i]) 
    upper_indices = np.where(pt>3 ) 
    lower_indices = np.setdiff1d(np.arange(len(pt)), upper_indices) 
    #cumulant2[i] = compute_four_particle_correlationv(store1, 2)
    cumulant_moment_POI2[i] = POI_order2(phi[lower_indices], phi[upper_indices],  2 ) 
    cumulant_moment_POI4[i] = POI_order4(phi[lower_indices], phi[upper_indices],  2 )
    cumulant_moment_2[i] = cumulant_2(phi,  2 ) 
    cumulant_moment_4[i] = cumulant_4(phi,  2 )
# cumulant2*weights/sum_weight
cumulant_moment_POI2 = np.real(cumulant_moment_POI2) 
cumulant_moment_POI4 = np.real(cumulant_moment_POI4) 
'''cumulant_moment_POI2*weights/ np.sum(weights )  
cumulant_moment_POI4*weights/ np.sum(weights )
 
cumulant_moment_2*weights/ np.sum(weights )  
cumulant_moment_4*weights/ np.sum(weights )''' 

 
end_time = time.time() 
pd.concat([
    pd.Series(cumulant_moment_2, name=' azimuthal_correlation2'),
    pd.Series(cumulant_moment_4, name='azimuthal_correlation4'),
    pd.Series(cumulant_moment_POI2, name='POI_azimuthal_correlation2'),
    pd.Series(cumulant_moment_POI4, name='POI_azimuthal_correlation4')
#], axis=1).to_csv('highmomentum_corrilations.csv', index=False)
], axis=1).to_csv('highmomentum_corrilations.csv', index=False, mode = 'a')

print(f"Total runtime: {end_time - start_time} seconds") 

Total runtime: 59606.91398334503 seconds


In [2]:
import numpy as np
np.array([1,2,3])*np.array([1,2,3])/4 

array([0.25, 1.  , 2.25])